# Рекомендательная система фильмов

Данные: MovieLens (`ml-latest-small`). Ноутбук состоит из трёх частей:
1. **API (задание 4.1)** — функции, которые возвращают рекомендации;
2. **БД (дополнительно)** — SQLite, куда сохраняются оценки пользователя;
3. **Приложение (задание 4.2)** — графический интерфейс, который использует только функции API.

## 0. Установка библиотек (после установки перезапустите ядро: Restart)

In [ ]:
%pip install -q pandas scikit-learn ipywidgets

## 1. Загрузка данных и база данных

In [ ]:
import os, ssl, zipfile, sqlite3, urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- данные MovieLens (скачиваются один раз) ---
if not os.path.exists('ml-latest-small'):
    url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
    try:
        data = urllib.request.urlopen(url).read()
    except Exception:                                   # на Windows часто ошибка сертификата SSL
        data = urllib.request.urlopen(url, context=ssl._create_unverified_context()).read()
    open('ml.zip', 'wb').write(data)
    zipfile.ZipFile('ml.zip').extractall('.')

movies = pd.read_csv('ml-latest-small/movies.csv')                                  # movieId, title, genres
ratings = pd.read_csv('ml-latest-small/ratings.csv')[['userId', 'movieId', 'rating']]

# --- БД: сюда сохраняются оценки, поставленные в приложении ---
conn = sqlite3.connect('ratings.db', check_same_thread=False)
conn.execute('CREATE TABLE IF NOT EXISTS my_ratings(userId INT, movieId INT, rating REAL, PRIMARY KEY(userId, movieId))')

def all_ratings():
    # оценки из датасета + оценки из БД (оценки из БД главнее)
    mine = pd.read_sql('SELECT * FROM my_ratings', conn)
    data = pd.concat([ratings, mine]).drop_duplicates(['userId', 'movieId'], keep='last')
    return data.astype({'userId': int, 'movieId': int, 'rating': float})

print(len(movies), 'фильмов,', len(ratings), 'оценок')

## 2. API (задание 4.1)

Сначала строится матрица «пользователь × фильм» и матрицы сходства, затем 4 функции-рекомендации.

In [ ]:
def build():
    # Строим модель по всем оценкам (датасет + БД)
    global R, mean, C, user_sim, item_sim
    R = all_ratings().pivot(index='userId', columns='movieId', values='rating')   # матрица оценок
    mean = R.mean(axis=1)                                                         # средняя оценка пользователя
    C = R.sub(mean, axis=0).fillna(0)                                             # оценки минус среднее, пропуски = 0
    # сходство пользователей
    user_sim = pd.DataFrame(cosine_similarity(C), index=C.index, columns=C.index)
    # сходство фильмов (только фильмы, у которых >= 50 оценок)
    popular = R.count()[lambda s: s >= 50].index
    item_sim = pd.DataFrame(cosine_similarity(C[popular].T), index=popular, columns=popular)

def with_titles(scores):
    # превращает Series {movieId: балл} в таблицу с названиями
    df = scores.rename('score').round(2).to_frame().join(movies.set_index('movieId')[['title', 'genres']])
    return df[['title', 'genres', 'score']]

def top_n(scores, user_id, n=10):
    # выбираем n лучших фильмов, которые пользователь ещё не оценивал
    scores = scores.dropna()
    scores = scores[~scores.index.isin(R.loc[user_id].dropna().index)]
    return with_titles(scores.nlargest(n))

# ---------- 1. Топ-10 популярных ----------
def get_top10():
    stats = pd.DataFrame({'count': R.count(), 'avg': R.mean()})
    best = stats[stats['count'] >= 50]['avg'].nlargest(10)       # средняя оценка среди фильмов с >= 50 оценок
    return with_titles(best)

# ---------- 2а. Коллаборативная фильтрация по пользователям ----------
def predict_user_based(user_id, k=20):
    neigh = user_sim[user_id].drop(user_id).nlargest(k)          # k самых похожих пользователей
    num = neigh.dot(C.loc[neigh.index])                          # сумма (сходство * оценка соседа)
    den = neigh.dot(R.loc[neigh.index].notna().astype(float))    # сумма сходств тех, кто оценил фильм
    return (mean[user_id] + num / den.replace(0, np.nan)).clip(0.5, 5)

def recommend_user_based(user_id):
    return top_n(predict_user_based(user_id), user_id)

# ---------- 2б. Коллаборативная фильтрация по фильмам ----------
def predict_item_based(user_id):
    rated = R.loc[user_id, item_sim.index].notna().astype(float)  # какие фильмы оценил пользователь
    num = item_sim.dot(C.loc[user_id, item_sim.index])            # сходство с оценёнными * оценка
    den = item_sim.abs().dot(rated)
    return (mean[user_id] + num / den.replace(0, np.nan)).clip(0.5, 5)

def recommend_item_based(user_id):
    return top_n(predict_item_based(user_id), user_id)

# ---------- 3. По контенту (жанры) ----------
tfidf = TfidfVectorizer(token_pattern=r'[^|]+').fit_transform(movies.genres)    # жанры -> числа

def recommend_content(movie_id):
    i = movies.index[movies.movieId == movie_id][0]
    sim = pd.Series(cosine_similarity(tfidf[i], tfidf).ravel(), index=movies.movieId)   # похожесть жанров
    sim = sim.drop(movie_id)
    sim = sim[R.count().reindex(sim.index).fillna(0) >= 20]       # только фильмы с >= 20 оценок
    return with_titles(sim.nlargest(10))

# ---------- 4. Персональные рекомендации ----------
def recommend_personal(user_id):
    if user_id not in R.index:                                    # новый пользователь без оценок
        return get_top10()
    both = pd.concat([predict_user_based(user_id), predict_item_based(user_id)], axis=1).mean(axis=1)
    return top_n(both, user_id)                                   # среднее двух методов

# ---------- Оценка фильма (запись в БД) ----------
def save_rating(user_id, movie_id, rating):
    conn.execute('INSERT OR REPLACE INTO my_ratings VALUES(?,?,?)', (user_id, movie_id, rating))
    conn.commit()
    build()                                                       # пересчитываем модель с новой оценкой

def find_movies(text):
    return movies[movies.title.str.contains(text, case=False, regex=False)].head(30)

# ---------- Справка ----------
HELP = """
Команды и параметры:
  get_top10()                     - топ-10 популярных фильмов. Параметров нет.
  recommend_user_based(user_id)   - коллаборативная фильтрация по пользователям. user_id - номер пользователя.
  recommend_item_based(user_id)   - коллаборативная фильтрация по фильмам. user_id - номер пользователя.
  recommend_content(movie_id)     - 10 фильмов, похожих по жанрам. movie_id - номер фильма.
  recommend_personal(user_id)     - персональные рекомендации (среднее двух методов CF). user_id - номер пользователя.
  save_rating(user_id, movie_id, rating) - сохранить оценку в БД. rating от 0.5 до 5.
  find_movies(text)               - поиск фильма по части названия.
"""

build()
get_top10()

### Проверка API

In [ ]:
print('Пользователь по оценкам (user 1):'); display(recommend_user_based(1))
print('По фильмам (user 1):');            display(recommend_item_based(1))
first = get_top10().index[0]   # id самого популярного фильма
print('По контенту (похожие на фильм', first, '):'); display(recommend_content(first))
print('Персональные (user 1):');          display(recommend_personal(1))

## 3. Приложение с графическим интерфейсом (задание 4.2)

1. Выберите команду и нажмите **«Показать»**.
2. Чтобы оценить фильм: найдите его по названию, выберите, поставьте оценку и нажмите **«Оценить»** (сохраняется в БД).
3. Кнопка **«Справка»** показывает команды и параметры.

In [ ]:
cmd = widgets.Dropdown(description='Команда:', layout=widgets.Layout(width='550px'), options=[
    ('1. Топ-10 популярных', 'top'),
    ('2а. Коллаборативная фильтрация: по пользователям', 'user'),
    ('2б. Коллаборативная фильтрация: по фильмам', 'item'),
    ('3. По контенту (нужен выбранный фильм)', 'content'),
    ('4. Персональные рекомендации', 'personal')])
user = widgets.BoundedIntText(value=1, min=1, max=10**6, description='ID пользователя:', style={'description_width': 'initial'})
search = widgets.Text(description='Поиск фильма:', placeholder='часть названия')
find_btn = widgets.Button(description='Найти')
movie = widgets.Dropdown(description='Фильм:', layout=widgets.Layout(width='550px'))
show_btn = widgets.Button(description='Показать', button_style='primary')
rate = widgets.FloatSlider(value=4, min=0.5, max=5, step=0.5, description='Оценка:')
rate_btn = widgets.Button(description='Оценить', button_style='success')
help_btn = widgets.Button(description='Справка')
out = widgets.Output()

def on_find(b):
    movie.options = [(row.title, row.movieId) for row in find_movies(search.value).itertuples()]

def on_show(b):
    with out:
        clear_output()
        c, u = cmd.value, user.value
        if c == 'top':
            display(get_top10())
        elif c == 'content':
            if movie.value is None: print('Сначала найдите и выберите фильм.')
            else: display(recommend_content(movie.value))
        elif u not in R.index:
            print('Пользователь', u, 'не найден или у него нет оценок. Оцените несколько фильмов.')
            display(get_top10())
        else:
            display({'user': recommend_user_based, 'item': recommend_item_based, 'personal': recommend_personal}[c](u))

def on_rate(b):
    with out:
        clear_output()
        if movie.value is None: print('Сначала найдите и выберите фильм.')
        else:
            save_rating(user.value, movie.value, rate.value)
            print('Сохранено в БД: пользователь', user.value, ', фильм', movie.label, ', оценка', rate.value)

def on_help(b):
    with out:
        clear_output(); print(HELP)

find_btn.on_click(on_find); show_btn.on_click(on_show); rate_btn.on_click(on_rate); help_btn.on_click(on_help)

display(widgets.VBox([cmd, user, show_btn,
                      widgets.HBox([search, find_btn]), movie, rate,
                      widgets.HBox([rate_btn, help_btn]), out]))